In [1]:
# %%
"""
📌 Notebook: 04_dataset_creation.ipynb

🎯 Purpose:
Convert feature dataset into sequences suitable for deep learning models.

Steps:
1. Global normalization (IMPORTANT FIX)
2. Sliding window sequence creation
3. Train-test split
4. Save processed arrays
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("✅ Libraries loaded.")

✅ Libraries loaded.


In [2]:
# %%
# Load dataset
input_path = "../processed_data/feature_dataset.csv"
df = pd.read_csv(input_path)

print(f"✅ Dataset loaded. Total rows: {len(df)}")

✅ Dataset loaded. Total rows: 2750848


In [3]:
# %%
"""
🔹 Step 1: Feature Selection
"""

feature_cols = [
    'X_Acc', 'Y_Acc', 'Z_Acc',
    'X_Gyro', 'Y_Gyro', 'Z_Gyro',
    'acc_mag', 'gyro_mag', 'jerk'
]

print(f"Using {len(feature_cols)} features.")

Using 9 features.


In [4]:
# %%
"""
🔹 Step 2: GLOBAL NORMALIZATION (CRITICAL FIX)

Fit scaler on entire dataset, not per trip.
"""

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print("✅ Global normalization applied.")

✅ Global normalization applied.


In [5]:
# %%
"""
🔹 Step 3: Create sequences using sliding window
"""

window_size = 100
step_size = 20

X, y = [], []

# Process each trip
for tid in df['trip_id'].unique():
    trip_data = df[df['trip_id'] == tid]
    
    if len(trip_data) < window_size:
        continue
    
    features = trip_data[feature_cols].values
    label = trip_data['rating'].iloc[0]
    
    for i in range(0, len(features) - window_size, step_size):
        X.append(features[i : i + window_size])
        y.append(label)

X = np.array(X).astype('float32')
y = np.array(y).astype('int32')

print(f"✅ Windowing complete")
print(f"Total windows: {len(X)}")
print(f"Shape: {X.shape}")  # (samples, 50, features)
print(f"Labels: {np.unique(y)}")

✅ Windowing complete
Total windows: 136658
Shape: (136658, 100, 9)
Labels: [1 2 3 4 5]


In [6]:
# %%
"""
🔹 Step 4: Train-Test Split
"""

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 109326
Testing samples: 27332


In [7]:
# %%
"""
💾 Step 5: Save datasets
"""

output_folder = "../processed_data"

np.save(f"{output_folder}/X_train.npy", X_train)
np.save(f"{output_folder}/X_test.npy", X_test)
np.save(f"{output_folder}/y_train.npy", y_train)
np.save(f"{output_folder}/y_test.npy", y_test)

print("✅ Dataset saved successfully.")

✅ Dataset saved successfully.
